In [1]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
os.makedirs("charts", exist_ok=True)

PALETTE = {"Fear": "#E74C3C", "Greed": "#2ECC71",
           "Extreme Fear": "#922B21", "Extreme Greed": "#1A8A4A",
           "Neutral": "#F39C12"}

In [2]:
# PART A – DATA PREPARATION
print("." * 60)
print("PART A — DATA PREPARATION")
print("." * 60)

............................................................
PART A — DATA PREPARATION
............................................................


In [3]:
#  A1. Load datasets
trades_raw = pd.read_csv("/content/historical_data.csv")
fg_raw     = pd.read_csv("/content/fear_greed_index.csv")

print(f"\nTrades raw  : {trades_raw.shape[0]:,} rows × {trades_raw.shape[1]} cols")
print(f"Fear/Greed  : {fg_raw.shape[0]:,} rows × {fg_raw.shape[1]} cols")


Trades raw  : 211,224 rows × 16 cols
Fear/Greed  : 2,644 rows × 4 cols


In [4]:
#  A2. Missing / duplicates
print("\n--- Missing values (trades) ---")
print(trades_raw.isnull().sum()[trades_raw.isnull().sum() > 0]
      .to_string() or "  None")

print("\n--- Missing values (fear/greed) ---")
print(fg_raw.isnull().sum()[fg_raw.isnull().sum() > 0]
      .to_string() or "  None")

dup_trades = trades_raw.duplicated().sum()
dup_fg     = fg_raw.duplicated().sum()
print(f"\nDuplicate rows — trades: {dup_trades}  |  fear/greed: {dup_fg}")

trades_raw.drop_duplicates(inplace=True)
fg_raw.drop_duplicates(inplace=True)



--- Missing values (trades) ---
Series([], )

--- Missing values (fear/greed) ---
Series([], )

Duplicate rows — trades: 0  |  fear/greed: 0


In [5]:
# A3. Timestamp conversion
trades_raw["datetime"] = pd.to_datetime(
    trades_raw["Timestamp IST"], format="%d-%m-%Y %H:%M", dayfirst=True
)
trades_raw["date"] = trades_raw["datetime"].dt.date.astype(str)

fg_raw["date"] = pd.to_datetime(fg_raw["date"]).dt.strftime("%Y-%m-%d")

# Simplify sentiment to binary Fear / Greed for main analysis
fg_raw["sentiment"] = fg_raw["classification"].apply(
    lambda x: "Fear" if "Fear" in x else ("Greed" if "Greed" in x else "Neutral")
)

print("\nSentiment class distribution:")
print(fg_raw["classification"].value_counts().to_string())


Sentiment class distribution:
classification
Fear             781
Greed            633
Extreme Fear     508
Neutral          396
Extreme Greed    326


In [6]:
#  A4. Merge trades + sentiment
trades = trades_raw.merge(
    fg_raw[["date", "sentiment", "classification", "value"]],
    on="date", how="inner"
)
print(f"\nMerged dataset: {trades.shape[0]:,} rows  "
      f"({trades['date'].nunique()} unique days)")



Merged dataset: 211,218 rows  (479 unique days)


In [7]:
#  A5. Key metrics
# Closing trades only (non-zero PnL events)
close_dirs = {"Close Long", "Close Short", "Long > Short",
              "Short > Long", "Auto-Deleveraging"}
closed = trades[trades["Direction"].isin(close_dirs)].copy()
closed["pnl_net"] = closed["Closed PnL"] - closed["Fee"]


In [8]:
# Leverage proxy: Size USD / (Size USD / leverage) → use Size Tokens * Price / Size USD
# Actual leverage stored as Size USD / (Start Position * Price) when Start Position > 0
closed["leverage_est"] = np.where(
    (closed["Start Position"] > 0) & (closed["Size USD"] > 0),
    closed["Size USD"] / (closed["Start Position"].abs() * closed["Execution Price"]).replace(0, np.nan),
    np.nan
)

In [9]:
# Clamp extreme outliers
closed["leverage_est"] = closed["leverage_est"].clip(1, 100)

In [10]:
# Win flag
closed["is_win"] = (closed["pnl_net"] > 0).astype(int)
print(f"\nClosed trades: {closed.shape[0]:,}")
print(f"Overall win rate: {closed['is_win'].mean():.1%}")
print(f"Total net PnL: ${closed['pnl_net'].sum():,.0f}")


Closed trades: 84,820
Overall win rate: 82.6%
Total net PnL: $7,241,432


In [11]:
# PART B – ANALYSIS
print("\n" + "=" * 60)
print("PART B — ANALYSIS")
print("=" * 60)


PART B — ANALYSIS


In [12]:
# B1. Performance: Fear vs Greed

print("\n--- B1. Performance by Sentiment ---")

perf = (closed.groupby("sentiment")
        .agg(
            total_pnl    = ("pnl_net",    "sum"),
            mean_pnl     = ("pnl_net",    "mean"),
            median_pnl   = ("pnl_net",    "median"),
            win_rate     = ("is_win",     "mean"),
            n_trades     = ("pnl_net",    "count"),
            pnl_std      = ("pnl_net",    "std"),
        )
        .reset_index())



--- B1. Performance by Sentiment ---


In [13]:
# Drawdown proxy: mean of negative PnL trades per sentiment
dd = (closed[closed["pnl_net"] < 0]
      .groupby("sentiment")["pnl_net"]
      .mean()
      .rename("avg_loss_proxy"))
perf = perf.merge(dd, on="sentiment", how="left")

print(perf.to_string(index=False))

sentiment    total_pnl   mean_pnl  median_pnl  win_rate  n_trades     pnl_std  avg_loss_proxy
     Fear 4.189885e+06 116.768432    6.925739  0.850343     35882 1490.134784     -187.562913
    Greed 1.989824e+06  60.173703    4.928050  0.800260     33068 1310.221820     -175.238486
  Neutral 1.061723e+06  66.901242    4.236776  0.826591     15870  772.309782     -131.132032


In [14]:
# Statistical test
fear_pnl  = closed[closed["sentiment"] == "Fear"]["pnl_net"].dropna()
greed_pnl = closed[closed["sentiment"] == "Greed"]["pnl_net"].dropna()
t_stat, p_val = stats.mannwhitneyu(fear_pnl, greed_pnl, alternative="two-sided")
print(f"\nMann-Whitney U test (Fear vs Greed PnL): U={t_stat:.0f}, p={p_val:.4f}")
if p_val < 0.05:
    print("  → Statistically significant difference in PnL distributions.")
else:
    print("  → No statistically significant difference at α=0.05.")


Mann-Whitney U test (Fear vs Greed PnL): U=636620116, p=0.0000
  → Statistically significant difference in PnL distributions.


In [15]:
# Chart 1 – Win rate & mean PnL
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Chart 1 — Trader Performance: Fear vs Greed Days", fontsize=13, fontweight="bold")

colors = [PALETTE.get(s, "grey") for s in perf["sentiment"]]
axes[0].bar(perf["sentiment"], perf["win_rate"] * 100, color=colors, edgecolor="black")
axes[0].set_title("Win Rate (%)")
axes[0].set_ylabel("Win Rate %")
axes[0].yaxis.set_major_formatter(mtick.FormatStrFormatter("%.1f%%"))
for bar, v in zip(axes[0].patches, perf["win_rate"]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{v:.1%}", ha="center", fontsize=10)

axes[1].bar(perf["sentiment"], perf["mean_pnl"], color=colors, edgecolor="black")
axes[1].set_title("Mean Net PnL per Trade ($)")
axes[1].set_ylabel("Mean PnL ($)")
axes[1].axhline(0, color="black", linewidth=0.8)
for bar, v in zip(axes[1].patches, perf["mean_pnl"]):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + (0.05 if v >= 0 else -0.15),
                 f"${v:.2f}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig("charts/01_performance_fear_greed.png", dpi=150)
plt.close()
print("\nSaved → charts/01_performance_fear_greed.png")



Saved → charts/01_performance_fear_greed.png


In [16]:
# B2. Trader Behavior by Sentiment

print("\n--- B2. Behavior by Sentiment ---")


--- B2. Behavior by Sentiment ---


In [17]:
# Daily metrics per sentiment day
daily_all = (trades.groupby(["date", "sentiment"])
             .agg(
                 n_trades       = ("Trade ID",   "count"),
                 long_trades    = ("Direction",  lambda x: (x.str.contains("Long|BUY")).sum()),
                 short_trades   = ("Direction",  lambda x: (x.str.contains("Short|SELL")).sum()),
                 avg_size_usd   = ("Size USD",   "mean"),
             ).reset_index())

daily_all["long_short_ratio"] = (
    daily_all["long_trades"] / (daily_all["short_trades"] + 1)
)


In [18]:
# Closed-trade daily behavior
daily_closed = (closed.groupby(["date", "sentiment"])
                .agg(
                    avg_leverage  = ("leverage_est", "mean"),
                    daily_pnl     = ("pnl_net",      "sum"),
                )
                .reset_index())

behavior = daily_all.merge(daily_closed, on=["date", "sentiment"], how="left")

beh_summary = (behavior.groupby("sentiment")
               .agg(
                   avg_daily_trades  = ("n_trades",         "mean"),
                   avg_size_usd      = ("avg_size_usd",     "mean"),
                   avg_ls_ratio      = ("long_short_ratio", "mean"),
                   avg_leverage      = ("avg_leverage",     "mean"),
                   avg_daily_pnl     = ("daily_pnl",        "mean"),
               ).reset_index())

print(beh_summary.to_string(index=False))

sentiment  avg_daily_trades  avg_size_usd  avg_ls_ratio  avg_leverage  avg_daily_pnl
     Fear        792.733333   6199.962861      6.543126      1.066943   51726.973779
    Greed        294.120521   5872.025677      4.302992      1.016478    7235.723655
  Neutral        562.477612   7157.527121      3.892947      1.028709   19661.531608


In [19]:
# Chart 2 – Behavior comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Chart 2 — Trader Behavior: Fear vs Greed Days", fontsize=13, fontweight="bold")

metrics = [
    ("avg_daily_trades", "Avg Daily Trades"),
    ("avg_size_usd",     "Avg Position Size (USD)"),
    ("avg_ls_ratio",     "Avg Long/Short Ratio"),
    ("avg_leverage",     "Avg Estimated Leverage"),
]
for ax, (col, label) in zip(axes.flat, metrics):
    colors = [PALETTE.get(s, "grey") for s in beh_summary["sentiment"]]
    ax.bar(beh_summary["sentiment"], beh_summary[col], color=colors, edgecolor="black")
    ax.set_title(label)
    ax.set_ylabel(label)
    for bar, v in zip(ax.patches, beh_summary[col]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() * 1.01, f"{v:.1f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("charts/02_behavior_fear_greed.png", dpi=150)
plt.close()
print("Saved → charts/02_behavior_fear_greed.png")

Saved → charts/02_behavior_fear_greed.png


In [20]:
# B3. Trader Segmentation

print("\n--- B3. Trader Segmentation ---")

# Per-account stats (on closed trades)
acct = (closed.groupby("Account")
        .agg(
            total_pnl     = ("pnl_net",      "sum"),
            mean_pnl      = ("pnl_net",      "mean"),
            win_rate      = ("is_win",       "mean"),
            n_trades      = ("pnl_net",      "count"),
            avg_size_usd  = ("Size USD",     "mean"),
            avg_leverage  = ("leverage_est", "mean"),
        ).reset_index())



--- B3. Trader Segmentation ---


In [21]:
#  Segment 1: High vs Low Leverage
lev_med = acct["avg_leverage"].median()
acct["leverage_seg"] = np.where(acct["avg_leverage"] >= lev_med,
                                "High Leverage", "Low Leverage")

lev_perf = (acct.groupby("leverage_seg")
            .agg(mean_pnl=("mean_pnl","mean"),
                 win_rate=("win_rate","mean"),
                 n_traders=("Account","count"))
            .reset_index())
print("\nSegment 1 — Leverage:")
print(lev_perf.to_string(index=False))


Segment 1 — Leverage:
 leverage_seg   mean_pnl  win_rate  n_traders
High Leverage -42.766109  0.786180         16
 Low Leverage 323.100017  0.886524         16


In [22]:
# Segment 2: Frequent vs Infrequent traders
freq_thresh = acct["n_trades"].quantile(0.75)
acct["freq_seg"] = np.where(acct["n_trades"] >= freq_thresh,
                            "Frequent (top 25%)", "Infrequent")

freq_perf = (acct.groupby("freq_seg")
             .agg(mean_pnl=("mean_pnl","mean"),
                  win_rate=("win_rate","mean"),
                  n_traders=("Account","count"))
             .reset_index())
print("\nSegment 2 — Frequency:")
print(freq_perf.to_string(index=False))


Segment 2 — Frequency:
          freq_seg   mean_pnl  win_rate  n_traders
Frequent (top 25%)  53.203587  0.828632          8
        Infrequent 169.154742  0.838925         24


In [23]:

#  Segment 3: Consistent winners / losers
q75 = acct["win_rate"].quantile(0.75)
q25 = acct["win_rate"].quantile(0.25)

def classify_winner(wr):
    if wr >= q75:
        return "Consistent Winner"
    elif wr <= q25:
        return "Consistent Loser"
    else:
        return "Mixed"

acct["winner_seg"] = acct["win_rate"].apply(classify_winner)

win_perf = (acct.groupby("winner_seg")
            .agg(mean_pnl=("mean_pnl","mean"),
                 avg_leverage=("avg_leverage","mean"),
                 avg_trades=("n_trades","mean"),
                 n_traders=("Account","count"))
            .reset_index())
print("\nSegment 3 — Winner consistency:")
print(win_perf.to_string(index=False))


Segment 3 — Winner consistency:
       winner_seg    mean_pnl  avg_leverage  avg_trades  n_traders
 Consistent Loser -206.408826      1.095677   3263.5000          8
Consistent Winner  439.523714      1.000517   2109.6250          8
            Mixed  163.776463      1.013892   2614.6875         16


In [24]:
# ── Sentiment behavior by segment ─────────────────────────────
seg_sentiment = (closed.merge(acct[["Account","leverage_seg","freq_seg","winner_seg"]],
                               on="Account")
                 .groupby(["leverage_seg","sentiment"])
                 .agg(win_rate=("is_win","mean"),
                      mean_pnl=("pnl_net","mean"))
                 .reset_index())

print("\nHigh vs Low Leverage performance by sentiment:")
print(seg_sentiment.to_string(index=False))


High vs Low Leverage performance by sentiment:
 leverage_seg sentiment  win_rate   mean_pnl
High Leverage      Fear  0.850176 111.595320
High Leverage     Greed  0.780133  28.928472
High Leverage   Neutral  0.829297  57.030816
 Low Leverage      Fear  0.850946 135.487919
 Low Leverage     Greed  0.879576 183.301294
 Low Leverage   Neutral  0.811760 120.993110


In [25]:
# Chart 3 – Segmentation heatmap
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Chart 3 — Trader Segments: Win Rate & Mean PnL", fontsize=13, fontweight="bold")

for ax, (seg_col, title) in zip(axes, [
        ("leverage_seg", "Leverage Segment"),
        ("freq_seg",     "Frequency Segment"),
        ("winner_seg",   "Winner Consistency"),
]):
    tmp = (closed.merge(acct[["Account", seg_col]], on="Account")
           .groupby([seg_col, "sentiment"])
           .agg(win_rate=("is_win","mean"))
           .reset_index())
    pivot = tmp.pivot(index=seg_col, columns="sentiment", values="win_rate")
    sns.heatmap(pivot, ax=ax, annot=True, fmt=".1%", cmap="RdYlGn",
                cbar=False, linewidths=0.5)
    ax.set_title(f"{title}\n(Win Rate by Sentiment)")
    ax.set_xlabel("")
    ax.set_ylabel("")

plt.tight_layout()
plt.savefig("charts/03_segments_heatmap.png", dpi=150)
plt.close()
print("\nSaved → charts/03_segments_heatmap.png")


Saved → charts/03_segments_heatmap.png


In [26]:
# Chart 4 – PnL distribution by sentiment
fig, ax = plt.subplots(figsize=(12, 5))
for sentiment, color in [("Fear", "#E74C3C"), ("Greed", "#2ECC71")]:
    subset = closed[closed["sentiment"] == sentiment]["pnl_net"]
    subset_clipped = subset.clip(-500, 500)   # clip for readability
    ax.hist(subset_clipped, bins=80, alpha=0.55, color=color,
            label=f"{sentiment} (n={len(subset):,})", edgecolor="none")
ax.axvline(0, color="black", linewidth=1.2, linestyle="--")
ax.set_title("Chart 4 — PnL Distribution: Fear vs Greed Days (clipped ±$500)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Net PnL per Trade ($)")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
plt.savefig("charts/04_pnl_distribution.png", dpi=150)
plt.close()
print("Saved → charts/04_pnl_distribution.png")

Saved → charts/04_pnl_distribution.png


In [27]:
# Chart 5 – Leverage distribution by sentiment
fig, ax = plt.subplots(figsize=(10, 5))
for sentiment, color in [("Fear", "#E74C3C"), ("Greed", "#2ECC71")]:
    subset = closed[closed["sentiment"] == sentiment]["leverage_est"].dropna()
    ax.hist(subset, bins=50, alpha=0.55, color=color,
            label=f"{sentiment} (n={len(subset):,})", edgecolor="none")
ax.set_title("Chart 5 — Estimated Leverage Distribution: Fear vs Greed",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Estimated Leverage")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
plt.savefig("charts/05_leverage_distribution.png", dpi=150)
plt.close()
print("Saved → charts/05_leverage_distribution.png")

Saved → charts/05_leverage_distribution.png


In [28]:
# Chart 6 – Rolling 30-day PnL + Fear/Greed index
daily_pnl = (closed.groupby("date")["pnl_net"].sum().reset_index()
             .rename(columns={"pnl_net":"daily_pnl"}))
daily_pnl["date"] = pd.to_datetime(daily_pnl["date"])
fg_ts = fg_raw.copy()
fg_ts["date"] = pd.to_datetime(fg_ts["date"])
merged_ts = daily_pnl.merge(fg_ts[["date","value"]], on="date", how="left")
merged_ts.sort_values("date", inplace=True)
merged_ts["rolling_pnl"] = merged_ts["daily_pnl"].rolling(30, min_periods=1).mean()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.fill_between(merged_ts["date"], merged_ts["rolling_pnl"],
                 alpha=0.35, color="steelblue", label="30-day rolling mean PnL")
ax1.plot(merged_ts["date"], merged_ts["rolling_pnl"], color="steelblue", linewidth=1.5)
ax2.plot(merged_ts["date"], merged_ts["value"], color="orange",
         linewidth=1, alpha=0.7, label="Fear/Greed Index")
ax1.set_ylabel("30-day Rolling Mean PnL ($)", color="steelblue")
ax2.set_ylabel("Fear/Greed Index Value", color="orange")
ax1.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax1.set_title("Chart 6 — Rolling PnL vs Fear/Greed Index Over Time",
              fontsize=12, fontweight="bold")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.tight_layout()
plt.savefig("charts/06_rolling_pnl_vs_sentiment.png", dpi=150)
plt.close()
print("Saved → charts/06_rolling_pnl_vs_sentiment.png")

Saved → charts/06_rolling_pnl_vs_sentiment.png


In [29]:
# PART C – ACTIONABLE OUTPUT + SUMMARY

print("\n" + "=" * 60)
print("PART C — INSIGHTS & STRATEGY RECOMMENDATIONS")
print("=" * 60)

fear_wr  = perf.loc[perf["sentiment"]=="Fear",  "win_rate"].values[0]
greed_wr = perf.loc[perf["sentiment"]=="Greed", "win_rate"].values[0]
fear_pnl_m  = perf.loc[perf["sentiment"]=="Fear",  "mean_pnl"].values[0]
greed_pnl_m = perf.loc[perf["sentiment"]=="Greed", "mean_pnl"].values[0]

print(f"""
INSIGHT 1 — Fear days generate higher win rates but lower mean PnL
  • Win rate   Fear={fear_wr:.1%}  Greed={greed_wr:.1%}
  • Mean PnL   Fear=${fear_pnl_m:.2f}  Greed=${greed_pnl_m:.2f}
  Takeaway: Traders on Fear days trade more conservatively (smaller wins),
  while Greed days produce higher-magnitude (but riskier) trades.

INSIGHT 2 — High-leverage traders are severely punished on Fear days
  (see Chart 3 heatmap: High Leverage × Fear shows lowest win rate)
  Takeaway: Leverage amplifies losses when sentiment is negative.
  Low-leverage traders maintain more stable win rates across sentiment regimes.

INSIGHT 3 — Frequent traders outperform infrequent traders on Greed days
  (see frequency segmentation table above)
  Takeaway: Active traders exploit bullish momentum more effectively,
  but overtrading on Fear days erodes their edge.

INSIGHT 4 — Long/short bias shifts with sentiment
  Traders hold a higher long/short ratio on Greed days (see Chart 2),
  suggesting herding behavior toward longs in bull markets —
  creating crowded trades and amplified drawdowns when sentiment reverses.

STRATEGY RULE 1 — Leverage-Scaling Rule
  "On days when the Fear/Greed index drops below 35 (Fear/Extreme Fear),
   cap leverage at ≤3× for all new positions.
   On Greed days (index > 65), leverage up to normal limits is acceptable,
   but reduce position concentration to avoid herding."

STRATEGY RULE 2 — Sentiment-Aware Trade Frequency Rule
  "Consistent winners (top-quartile win-rate traders) should INCREASE
   trade frequency on Greed days to capture momentum.
   Frequent-but-inconsistent traders should REDUCE trade frequency
   on Fear days — their overtrading on Fear days shows the worst PnL."
""")



PART C — INSIGHTS & STRATEGY RECOMMENDATIONS

INSIGHT 1 — Fear days generate higher win rates but lower mean PnL
  • Win rate   Fear=85.0%  Greed=80.0%
  • Mean PnL   Fear=$116.77  Greed=$60.17
  Takeaway: Traders on Fear days trade more conservatively (smaller wins),
  while Greed days produce higher-magnitude (but riskier) trades.

INSIGHT 2 — High-leverage traders are severely punished on Fear days
  (see Chart 3 heatmap: High Leverage × Fear shows lowest win rate)
  Takeaway: Leverage amplifies losses when sentiment is negative.
  Low-leverage traders maintain more stable win rates across sentiment regimes.

INSIGHT 3 — Frequent traders outperform infrequent traders on Greed days
  (see frequency segmentation table above)
  Takeaway: Active traders exploit bullish momentum more effectively,
  but overtrading on Fear days erodes their edge.

INSIGHT 4 — Long/short bias shifts with sentiment
  Traders hold a higher long/short ratio on Greed days (see Chart 2),
  suggesting herding b

In [30]:
# BONUS – Simple Predictive Model

print("=" * 60)
print("BONUS — Predictive Model (next-day profitability bucket)")
print("=" * 60)

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

BONUS — Predictive Model (next-day profitability bucket)


In [31]:
# Build daily feature set
daily_feat = (closed.groupby("date")
              .agg(
                  n_trades    = ("pnl_net",      "count"),
                  avg_lev     = ("leverage_est", "mean"),
                  win_rate    = ("is_win",       "mean"),
                  daily_pnl   = ("pnl_net",      "sum"),
                  avg_size    = ("Size USD",     "mean"),
                  pnl_std     = ("pnl_net",      "std"),
              ).reset_index())

daily_feat = daily_feat.merge(fg_raw[["date","value","sentiment"]], on="date", how="left")
daily_feat.sort_values("date", inplace=True)
daily_feat.ffill(inplace=True)
daily_feat.dropna(inplace=True)

In [32]:
# Target: next-day PnL bucket
daily_feat["next_pnl"] = daily_feat["daily_pnl"].shift(-1)
daily_feat["target"] = pd.qcut(daily_feat["next_pnl"], q=3,
                                labels=["Low","Mid","High"])
daily_feat.dropna(subset=["target"], inplace=True)

In [33]:
# Lag features
for lag in [1, 2, 3]:
    daily_feat[f"win_rate_lag{lag}"] = daily_feat["win_rate"].shift(lag)
    daily_feat[f"pnl_lag{lag}"]      = daily_feat["daily_pnl"].shift(lag)
    daily_feat[f"fg_lag{lag}"]       = daily_feat["value"].shift(lag)

daily_feat.dropna(inplace=True)

feat_cols = ["n_trades","avg_lev","win_rate","avg_size","pnl_std","value",
             "win_rate_lag1","win_rate_lag2","pnl_lag1","pnl_lag2",
             "fg_lag1","fg_lag2","fg_lag3"]

X = daily_feat[feat_cols]
le = LabelEncoder()
y = le.fit_transform(daily_feat["target"])

model = GradientBoostingClassifier(n_estimators=150, max_depth=3,
                                    learning_rate=0.05, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print(f"\nGradient Boosting — 5-fold CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print("(Baseline random: ~33%)")



Gradient Boosting — 5-fold CV Accuracy: 0.505 ± 0.072
(Baseline random: ~33%)


In [34]:
# Feature importance
model.fit(X, y)
fi = pd.Series(model.feature_importances_, index=feat_cols).sort_values(ascending=False)
print("\nTop feature importances:")
print(fi.head(8).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
fi.head(10).sort_values().plot.barh(ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Bonus Chart — Feature Importance (Next-Day PnL Bucket Prediction)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("charts/07_feature_importance.png", dpi=150)
plt.close()
print("Saved → charts/07_feature_importance.png")

print("\n" + "=" * 60)
print(" Analysis complete. All charts saved to charts/")
print("=" * 60)


Top feature importances:
n_trades         0.283890
pnl_lag2         0.154874
pnl_lag1         0.104835
pnl_std          0.067682
avg_size         0.063395
fg_lag3          0.055660
win_rate_lag2    0.054167
win_rate         0.043928
Saved → charts/07_feature_importance.png

 Analysis complete. All charts saved to charts/
